# QKD-Lab — BB84 eavesdropper detection (QC-NI1)

Runs inside the Qniverse hosted notebook. Part 1 is a full paired offline demo (no credits).
Part 2 submits the same circuits to a Qniverse CPU simulator and **spends real account credits** -
it is gated behind an explicit opt-in and will not run on a plain Run All.

Nothing on this machine is deleted or reset - if the repo is already cloned it is only updated (fast-forwarded).


In [ ]:
import os, subprocess, sys
REPO = 'https://github.com/Quantum-cdac/Independent-class'
DEST = '/home/jovyan/Independent-class'
if not os.path.isdir(DEST):
    subprocess.run(['git', 'clone', REPO, DEST], check=True)
else:
    subprocess.run(['git', '-C', DEST, 'pull', '--ff-only'], check=True)
sys.path.insert(0, DEST)
print('repo ready at', DEST)


## Part 1 - paired offline demo (local exact engine, no credits)

Same seed, same Alice/Bob choices in both runs - the ONLY thing that changes is the eavesdropper.


In [ ]:
from qkdlab.rounds import make_rounds, save_manifest
from qkdlab.engine import simulate_run
from qkdlab.protocol import sift

SEED, ROUNDS = 20261114, 48
for eve in (False, True):
    rounds = make_rounds(ROUNDS, eve, SEED)
    bob, ev = simulate_run(rounds, SEED)
    res = sift(rounds, bob, ev)
    state = 'HIGH ERROR RATE (possible eavesdropping or noise)' if res.high_error_rate else 'within expected range'
    print(f'eve={eve!s:5s} sifted={len(res.sift_indices):3d} errors={res.errors:3d} '
          f'QBER={res.qber:.2%} wilson95=[{res.wilson_low:.2%},{res.wilson_high:.2%}] -> {state}')


## Part 2 - Qniverse backend run (SPENDS CREDITS - explicit opt-in required)

Rounds are packed as independent qubits in one circuit, `shots=1` - Qniverse runs one circuit per job
and returns aggregate counts. Eve-off uses 1 qubit per round, Eve-on uses 2 (channel + Eve ancilla,
coherent intercept-resend).

**Each submitted job costs 5 credits.** This cell refuses to run unless you set
`I_AUTHORIZE_CREDIT_SPEND = True` yourself below. Checking the balance only informs the decision -
it is not authorization.


In [ ]:
EVE = True        # toggle the eavesdropper scenario
MAX_WIRES = 24    # batch cap; 12 rounds/job with Eve, 24 without
rounds = make_rounds(ROUNDS, EVE, SEED)
save_manifest(rounds, EVE, SEED, os.path.join(DEST, 'evidence', 'venue-manifest.json'))
from qkdlab.emit_qasm import split_batches
batches = list(split_batches(rounds, EVE, MAX_WIRES))
print(f'{len(batches)} job(s) needed - {5*len(batches)} credits total')

# Import-path safety: the bit-order convention must pass its asymmetric
# known-pattern self-test before any backend output is trusted.
# A LIVE import must ALSO pass verify_against_manifest on an Eve-off batch:
# Bob bits must reproduce the manifest exactly before any counts are trusted.
from qkdlab.parse_backend import selfcheck
selfcheck()
print('counts-key self-check passed (rightmost char = c[0], venue-confirmed on job 38024)')


In [ ]:
# Set this True ONLY if you accept spending 5 credits per job on this Qniverse account.
I_AUTHORIZE_CREDIT_SPEND = False

if not I_AUTHORIZE_CREDIT_SPEND:
    print('Credit spend not authorized - skipping backend submission. '
          'Set I_AUTHORIZE_CREDIT_SPEND = True and re-run this cell to proceed.')
else:
    from Qniverse.gates import *
    from Qniverse.QniverseCircuit import *
    from Qniverse.backend import Backend
    from Qniverse.api import run, fetch_job_result, credit_balance
    from qkdlab.parse_backend import parse_counts, rounds_from_counts

    token = 'PASTE_QNIVERSE_API_KEY_HERE'
    backend = Backend()
    backend.processor('CPU')
    backend.hpc('QACC_Cluster')
    backend.simulator('qasm_simulator')

    bob_bits, eve_bits = [], []
    for start, batch in batches:
        w = (2 if EVE else 1) * len(batch)
        qc = QniverseCircuit()
        q = QuantumRegister(w, 'q'); c = ClassicalRegister(w, 'c')
        qc.add_register(q); qc.add_register(c)
        for r, spec in enumerate(batch):
            qi = r * (2 if EVE else 1)
            if spec.alice_bit: qc.add_gate(x, q[qi])
            if spec.alice_basis: qc.add_gate(h, q[qi])
            if EVE:
                if spec.eve_basis: qc.add_gate(h, q[qi])
                qc.add_gate(cx, q[qi], q[qi+1])
                if spec.eve_basis: qc.add_gate(h, q[qi])
            if spec.bob_basis: qc.add_gate(h, q[qi])
            qc.add_gate(measure, q[qi], c[qi])
            if EVE: qc.add_gate(measure, q[qi+1], c[qi+1])
        print(f'submitting batch rounds {start}-{start+len(batch)-1} ({5} credits)...')
        job = run(qc, backend, 1, token=token, job_name=f'qkd-lab-rounds-{start}')
        counts = parse_counts(str(fetch_job_result(token, job)))
        b, e = rounds_from_counts(counts, len(batch), EVE)
        bob_bits += b; eve_bits += e
    if not EVE:
        from qkdlab.parse_backend import verify_against_manifest
        verify_against_manifest(rounds, bob_bits)
        print('venue verification gate passed: Bob bits reproduce the Eve-off manifest')
    print('balance left:', credit_balance(token))
    res = sift(rounds, bob_bits, eve_bits)
    state = 'HIGH ERROR RATE' if res.high_error_rate else 'within expected range'
    print(f'Qniverse run: sifted={len(res.sift_indices)} errors={res.errors} '
          f'QBER={res.qber:.2%} -> {state}')


## Test suite

32 tests: engine correctness, coherent-Eve statistics, an independent numpy density-matrix
cross-check of every choice combination and of the emitted QASM, sifting, emitters, parsing,
paired-seed reproducibility, CLI end-to-end.


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], cwd=DEST)
